In [1]:
# Does Search Even Work?
# ============================================================
#
# A good accuracy for SoRL confounds two things:
#   (a) a good POLICY model (pick abstraction that maximizes reward)
#   (b) a good REWARD model (given an abstraction, maximize reward)
#
# A powerful reward model can work with a COLLAPSED policy model
# (predict same abstraction all the time) — this is exactly what we observe:
#   - "Diversity degrades accuracy" = diverse policy hurts reward model
#   - "Accurate SoRL has collapsed vocabulary" = reward model compensates for bad policy
#
# This strongly indicates POLICY COLLAPSE — the search procedure in current
# SoRL never learns to explore. Related to:
#   - Incorrect Jacobi loss (not aligned with multi-step recursion)
#   - Lack of STE (no gradient through abstract token selection)
#   - Select-best optimizing reward, not policy
#
# Experiment: decouple by using an EXTERNAL reward model.
#   - Fixed, deterministic reward: target_abs = f(preceding NL context)
#   - Train only: abs_loss + per-iteration jacobi_loss + STE reward
#   - NO base_traj_loss — the model cannot "hedge" by ignoring abstractions
#
# If policy learns to match the external target → search mechanism works,
#   the problem was reward-policy confounding. Fix: GRPO / REINFORCE.
# If policy collapses despite known-optimal targets → search mechanism
#   itself is broken. Fix: fundamentally different policy optimization.

In [2]:
# 1. Need to ensure data loader provides meaningful data (random idx in vocab size)
#    this helps ensure the reward function motivates a diverse choice of asbtraction
# 2. Then we can properly probe whether search process is done correctly

In [1]:
import torch
import torch.nn.functional as F
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID, extract_and_sample
from sorl.neo_utils import infer_rythmic_insert_mask, insert_tokens
torch.set_float32_matmul_precision('high')

device = "cuda" if torch.cuda.is_available() else "cpu"

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID + 1, 16],  # 50257 NL tokens, 16 abstract tokens
    n_layer=4, n_head=4, n_embd=128,
    device=device,
)
model = GAT(gat_config)

NL_VOCAB = model.vocab_sizes[0].item()   # 50257
ABS_VOCAB = model.vocab_sizes[1].item()  # 16
K = 3
MEMORY_SPAN = 1792
ATTN_BS = 1792
MAX_ITER = 2

# ── Random NL data loader ──
# Uniform random tokens from [0, NL_VOCAB), excluding BOS_TOKEN_ID.
# Each document: BOS + doc_len random NL tokens.
# get_batch concatenates batch_size documents into a single row (1, total_len).

class RandomNLDataLoader:
    def __init__(self, nl_vocab, bos_token_id, doc_len=13, device="cpu"):
        self.nl_vocab = nl_vocab
        self.bos = bos_token_id
        self.doc_len = doc_len      # NL tokens per document (excl. BOS)
        self.device = device
        # valid NL token pool: [0, nl_vocab) minus BOS
        self.pool_size = nl_vocab - 1  # exclude BOS

    def _rand_doc(self):
        """BOS + doc_len uniform random NL tokens (no BOS inside)."""
        toks = torch.randint(0, self.pool_size, (self.doc_len,), device=self.device)
        # shift tokens >= BOS up by 1 so BOS is never sampled
        toks[toks >= self.bos] += 1
        return torch.cat([torch.tensor([self.bos], device=self.device), toks])

    def get_batch(self, batch_size):
        docs = [self._rand_doc() for _ in range(batch_size)]
        tokens = torch.cat(docs).unsqueeze(0)           # (1, total_len)
        doc_ids = torch.cat([torch.full((len(d),), i, device=self.device)
                             for i, d in enumerate(docs)]).unsqueeze(0)
        return tokens, doc_ids

train_loader = RandomNLDataLoader(NL_VOCAB, BOS_TOKEN_ID, doc_len=13, device=device)
val_loader   = RandomNLDataLoader(NL_VOCAB, BOS_TOKEN_ID, doc_len=13, device=device)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [7]:
# ── External reward: deterministic target for each abstract position ──

def compute_external_targets(seq, nl_vocab, abs_vocab, K_ratio):
    """
    For each abstract position, target = floor(mean(preceding K NL ids) * abs_vocab / nl_vocab).
    Returns (abs_positions, target_token_ids in full vocab space).
    """
    is_abs = seq >= nl_vocab
    abs_pos = is_abs.nonzero(as_tuple=True)[0]
    targets = []
    for p in abs_pos:
        start = max(0, p.item() - K_ratio)
        preceding = seq[start:p.item()]
        nl_only = preceding[preceding < nl_vocab]
        if len(nl_only) == 0:
            targets.append(0)
        else:
            t = int(nl_only.float().mean().item() * abs_vocab / nl_vocab)
            targets.append(max(0, min(t, abs_vocab - 1)))
    return abs_pos, torch.tensor(targets, device=seq.device) + nl_vocab


def external_reward(seq, nl_vocab, abs_vocab, K_ratio):
    """Scalar reward = -mean |selected_abs - target_abs|."""
    abs_pos, target_ids = compute_external_targets(seq, nl_vocab, abs_vocab, K_ratio)
    if len(abs_pos) == 0:
        return torch.tensor(0.0, device=seq.device)
    return -(seq[abs_pos].float() - target_ids.float()).abs().mean()

# ── Decomposed search: Jacobi rollouts → external-reward selection ──

@torch.no_grad()
def rollout_and_select(tokens, model, n, K_ratio, max_iterations,
                       memory_span, attn_blocksize, temperature):
    """
    N Jacobi rollouts, pick the one with highest EXTERNAL reward.
    Returns best_seq, reward, reward_gap, per-iteration history.
    """
    nl_vocab = model.vocab_sizes[0].item()
    abs_vocab = model.vocab_sizes[1].item()
    data_len = tokens.shape[1]
    insert_mask = infer_rythmic_insert_mask(tokens, K_ratio, model.vocab_sizes[0])
    expanded = insert_tokens(tokens, insert_mask, nl_vocab)[:, :data_len]
    repeat_data = expanded.repeat_interleave(n, dim=0)

    recursion_mask = (repeat_data >= nl_vocab)
    recursion_mask[:, 0] = False

    # Ensure temperature is a tensor for extract_and_sample
    if isinstance(temperature, (int, float)):
        temp_exp = torch.tensor(max(temperature, 0.01), device=tokens.device)
    elif isinstance(temperature, torch.Tensor) and temperature.ndim == 1:
        temp_exp = temperature.clamp(min=0.01).view(-1, 1).expand_as(repeat_data)
    else:
        temp_exp = temperature

    idx = repeat_data.clone()
    history = []
    for _ in range(max_iterations):
        _, logits = model.forward(idx, memory_span, attn_blocksize)
        idx = extract_and_sample(logits, idx, recursion_mask, model.vocab_sizes, temp_exp)
        history.append(idx.clone())

    # Score each rollout by EXTERNAL reward (not perplexity)
    rewards = torch.stack([external_reward(idx[i], nl_vocab, abs_vocab, K_ratio) for i in range(n)])
    best = rewards.argmax().item()

    best_seq = idx[best].unsqueeze(0)
    best_hist = [h[best].unsqueeze(0) for h in history]
    return best_seq, rewards[best], rewards.max() - rewards.min(), best_hist

@torch.no_grad()
def evaluate_model(model, val_loader, n_batches=50, batch_size=4):
    """
    Evaluate model on val_loader over n_batches.
    Returns:
      - match_rate: fraction of abstract tokens exactly matching external target
      - mean_reward: average external reward
      - abs_distribution: Counter of which abstract tokens the model produces
    """
    from collections import Counter
    total_correct = 0
    total_abs = 0
    total_reward = 0.0
    abs_counter = Counter()
    target_counter = Counter()

    for _ in range(n_batches):
        vt, _ = val_loader.get_batch(batch_size)
        vs, vr, _, _ = rollout_and_select(
            vt, model, n=1, K_ratio=K, max_iterations=MAX_ITER,
            memory_span=MEMORY_SPAN, attn_blocksize=ATTN_BS,
            temperature=0.01)

        ap, ti = compute_external_targets(vs[0], NL_VOCAB, ABS_VOCAB, K)
        if len(ap) > 0:
            preds = vs[0, ap]
            total_correct += (preds == ti).sum().item()
            total_abs += len(ap)
            total_reward += external_reward(vs[0], NL_VOCAB, ABS_VOCAB, K).item()
            for p in preds.tolist():
                abs_counter[p] += 1
            for t in ti.tolist():
                target_counter[t] += 1

    match_rate = total_correct / max(total_abs, 1)
    mean_reward = total_reward / n_batches

    print(f"=== Evaluation ({n_batches} batches, {total_abs} abstract tokens) ===")
    print(f"  match_rate = {match_rate*100:.1f}%")
    print(f"  mean_reward = {mean_reward:.2f}")
    print(f"  --- predicted distribution ---")
    for tok in sorted(abs_counter.keys()):
        pct = abs_counter[tok] / total_abs * 100
        print(f"    token {tok} ({tok - NL_VOCAB:2d}): {abs_counter[tok]:4d} ({pct:5.1f}%)")
    print(f"  --- target distribution ---")
    for tok in sorted(target_counter.keys()):
        pct = target_counter[tok] / total_abs * 100
        print(f"    token {tok} ({tok - NL_VOCAB:2d}): {target_counter[tok]:4d} ({pct:5.1f}%)")
    n_unique = len(abs_counter)
    print(f"  unique predicted: {n_unique}/16  |  unique targets: {len(target_counter)}/16")
    return match_rate, mean_reward, abs_counter, target_counter

In [8]:
# ── Loss functions: direct CE on external targets + jacobi ──

def loss_abs(seq, model, memory_span, attn_blocksize):
    """CE on abstract token positions — can the model predict p(a|context)?"""
    ppt, _ = model.forward(seq, memory_span, attn_blocksize)
    ppt = ppt.reshape(seq.shape[0], -1)
    is_abs = (seq[:, 1:] >= NL_VOCAB).float()
    bos = ((seq[:, :-1] != BOS_TOKEN_ID) & (seq[:, 1:] != BOS_TOKEN_ID)).float()
    mask = is_abs * bos
    return (ppt * mask).sum() / mask.sum().clamp(min=1)


def loss_jacobi(seq, model, history, memory_span, attn_blocksize):
    """
    Per-iteration jacobi loss — aligned with multi-step recursion.
    Only computes CE on abstract positions; non-abstract positions are ignored.
    Token NL_VOCAB (the mask/placeholder) is excluded from valid predictions.
    """
    total = torch.tensor(0.0, device=seq.device)
    for i, target_seq in enumerate(history):
        if i == 0:
            inp = target_seq.clone()
            inp[:, :][inp >= NL_VOCAB] = NL_VOCAB  # placeholder for abstract positions
        else:
            inp = history[i - 1]

        _, logits = model.forward(inp, memory_span, attn_blocksize)

        # Only look at abstract target positions
        is_abs = (target_seq[:, 1:] >= NL_VOCAB) & (target_seq[:, 1:] != NL_VOCAB)
        if is_abs.sum() == 0:
            continue

        abs_logits = logits[:, :-1, :].contiguous()
        # Mask NL tokens AND the placeholder token (NL_VOCAB)
        abs_logits[:, :, :NL_VOCAB + 1] = float('-inf')

        # Use a safe dummy target for non-abstract positions (NL_VOCAB+1 is valid abstract)
        tgt = target_seq[:, 1:].clone()
        tgt[~is_abs] = NL_VOCAB + 1  # valid abstract token, masked out below

        ce = F.cross_entropy(abs_logits.view(-1, abs_logits.size(-1)),
                             tgt.view(-1), reduction='none').view_as(tgt)
        total = total + (ce * is_abs.float()).sum() / is_abs.float().sum().clamp(min=1)
    return total / max(len(history), 1)


def loss_target_ce(seq, model, memory_span, attn_blocksize, K_ratio):
    """
    Abstraction CE loss
    """
    _, logits = model.forward(seq, memory_span, attn_blocksize)
    abs_pos, target_ids = compute_external_targets(seq[0], NL_VOCAB, ABS_VOCAB, K_ratio)
    if len(abs_pos) == 0:
        return torch.tensor(0.0, device=seq.device)

    # logits[:, t] predicts token at position t+1, so predict_pos = abs_pos - 1
    pred_pos = abs_pos - 1
    valid = pred_pos >= 0
    pred_pos = pred_pos[valid]
    target_ids = target_ids[valid]
    if len(pred_pos) == 0:
        return torch.tensor(0.0, device=seq.device)

    pred_logits = logits[0, pred_pos, :]  # (n, total_vocab)
    return F.cross_entropy(pred_logits, target_ids)

In [10]:
# ── Training loop ──
# Key metric: val_match% — fraction of abstract tokens exactly matching external target.
# If this converges toward 100%, the search mechanism works.
# If it stays near random (1/16 ≈ 6%), the search is broken.

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

batch_size = 4
n_rollouts = 4
temperatures = torch.tensor([0.0, 3.0, 3.0, 3.0], device=device)
alpha_tce = 1.0    # hard CE on external targets (main signal)
alpha_jac = 0.0    # jacobi consistency
num_steps = 1000

for step in range(num_steps):
    optimizer.zero_grad()
    tokens, _ = train_loader.get_batch(batch_size)

    # Search: Jacobi rollouts → pick best by EXTERNAL reward
    best_seq, ext_r, r_gap, hist = rollout_and_select(
        tokens, model, n=n_rollouts, K_ratio=K,
        max_iterations=MAX_ITER, memory_span=MEMORY_SPAN,
        attn_blocksize=ATTN_BS, temperature=temperatures)

    # Losses: hard CE on targets + jacobi consistency
    t_loss = loss_target_ce(best_seq, model, MEMORY_SPAN, ATTN_BS, K)
    j_loss = loss_jacobi(best_seq, model, hist, MEMORY_SPAN, ATTN_BS)

    loss = alpha_tce * t_loss + alpha_jac * j_loss
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        with torch.no_grad():
            vt, _ = val_loader.get_batch(batch_size)
            vs, vr, _, _ = rollout_and_select(
                vt, model, n=1, K_ratio=K, max_iterations=MAX_ITER,
                memory_span=MEMORY_SPAN, attn_blocksize=ATTN_BS,
                temperature=0.0)
            ap, ti = compute_external_targets(vs[0], NL_VOCAB, ABS_VOCAB, K)
            match = (vs[0, ap] == ti).float().mean().item() if len(ap) > 0 else 0.0
        print(f"step {step:4d} | loss={loss.item():.3f} "
              f"tce={t_loss.item():.3f} jac={j_loss.item():.3f} | "
              f"ext_r={ext_r.item():.2f} gap={r_gap.item():.2f} | "
              f"val_match={match*100:.1f}%")

step    0 | loss=8.954 tce=8.954 jac=2.271 | ext_r=-2.58 gap=2.25 | val_match=8.3%
step   10 | loss=6.630 tce=6.630 jac=1.916 | ext_r=-1.83 gap=1.33 | val_match=25.0%
step   20 | loss=4.526 tce=4.526 jac=1.800 | ext_r=-1.75 gap=2.50 | val_match=0.0%
step   30 | loss=3.573 tce=3.573 jac=1.714 | ext_r=-2.92 gap=2.33 | val_match=16.7%
step   40 | loss=3.018 tce=3.018 jac=1.759 | ext_r=-2.92 gap=1.42 | val_match=16.7%
step   50 | loss=3.173 tce=3.173 jac=1.709 | ext_r=-3.17 gap=1.58 | val_match=0.0%
step   60 | loss=2.740 tce=2.740 jac=1.832 | ext_r=-2.25 gap=2.25 | val_match=0.0%
step   70 | loss=2.303 tce=2.303 jac=1.821 | ext_r=-2.33 gap=2.00 | val_match=8.3%
step   80 | loss=2.608 tce=2.608 jac=1.886 | ext_r=-2.83 gap=0.67 | val_match=16.7%
step   90 | loss=2.479 tce=2.479 jac=1.704 | ext_r=-2.00 gap=0.92 | val_match=8.3%
step  100 | loss=2.277 tce=2.277 jac=1.781 | ext_r=-1.92 gap=1.75 | val_match=16.7%
step  110 | loss=2.381 tce=2.381 jac=1.566 | ext_r=-1.92 gap=1.50 | val_match=8.3%

In [11]:
evaluate_model(model, val_loader, n_batches=100, batch_size=4) # reinforce algorithm result is even worse than the best of N mechanism
# match rate 6.5%
# - much worse (despite having diverse vocabulary)

=== Evaluation (100 batches, 1200 abstract tokens) ===
  match_rate = 15.2%
  mean_reward = -2.12
  --- predicted distribution ---
    token 50264 ( 7): 1200 (100.0%)
  --- target distribution ---
    token 50257 ( 0):    1 (  0.1%)
    token 50258 ( 1):   10 (  0.8%)
    token 50259 ( 2):   29 (  2.4%)
    token 50260 ( 3):   43 (  3.6%)
    token 50261 ( 4):   88 (  7.3%)
    token 50262 ( 5):  110 (  9.2%)
    token 50263 ( 6):  159 ( 13.2%)
    token 50264 ( 7):  183 ( 15.2%)
    token 50265 ( 8):  162 ( 13.5%)
    token 50266 ( 9):  157 ( 13.1%)
    token 50267 (10):  107 (  8.9%)
    token 50268 (11):   80 (  6.7%)
    token 50269 (12):   38 (  3.2%)
    token 50270 (13):   23 (  1.9%)
    token 50271 (14):    8 (  0.7%)
    token 50272 (15):    2 (  0.2%)
  unique predicted: 1/16  |  unique targets: 16/16


(0.1525,
 -2.1199999952316286,
 Counter({50264: 1200}),
 Counter({50264: 183,
          50265: 162,
          50263: 159,
          50266: 157,
          50262: 110,
          50267: 107,
          50261: 88,
          50268: 80,
          50260: 43,
          50269: 38,
          50259: 29,
          50270: 23,
          50258: 10,
          50271: 8,
          50272: 2,
          50257: 1}))

#### Reinforcement learning method

In [4]:
@torch.no_grad()
def rollout_with_advantage(tokens, model, n, K_ratio, max_iterations,
                       memory_span, attn_blocksize, temperature):
    """
    N Jacobi rollouts → rewards → advantages (normalized).
    Returns: idx (n, L), rewards (n,), advantages (n,), history [list of (n, L)]
    """
    nl_vocab = model.vocab_sizes[0].item()
    abs_vocab = model.vocab_sizes[1].item()
    data_len = tokens.shape[1]
    insert_mask = infer_rythmic_insert_mask(tokens, K_ratio, model.vocab_sizes[0])
    expanded = insert_tokens(tokens, insert_mask, nl_vocab)[:, :data_len]
    repeat_data = expanded.repeat_interleave(n, dim=0)

    recursion_mask = (repeat_data >= nl_vocab)
    recursion_mask[:, 0] = False

    # ─── Temperature: match recursion() in gat_sim.py ───
    # Safest: pass scalar float or 0-dim tensor. Avoid 1D/2D tensors
    # which can cause issues with @torch.compile'd extract_and_sample.
    if isinstance(temperature, torch.Tensor):
        if temperature.ndim == 0:
            temp_val = max(temperature.item(), 0.01)
        elif temperature.ndim == 1:
            temp_val = max(temperature[0].item(), 0.01)
        else:
            temp_val = max(temperature.flatten()[0].item(), 0.01)
    else:
        temp_val = max(float(temperature), 0.01)
    temp_exp = torch.tensor(temp_val, device=tokens.device)

    idx = repeat_data.clone()
    history = []
    for _ in range(max_iterations):
        _, logits = model.forward(idx, memory_span, attn_blocksize)
        idx = extract_and_sample(logits, idx, recursion_mask, model.vocab_sizes, temp_exp)
        history.append(idx.clone())

    rewards = torch.stack([external_reward(idx[i], nl_vocab, abs_vocab, K_ratio) for i in range(n)])
    std = rewards.std().clamp(min=1e-8)
    advantages = (rewards - rewards.mean()) / std

    return idx, rewards, advantages, history

In [5]:
# ── REINFORCE training loop ──
# 
# 1. Sample N rollouts (no grad) → rewards, advantages
# 2. Replay each iteration with grad → log π(sampled_token | state)
# 3. Loss = -advantage × Σ_t log π(a_t | s_t)   (policy gradient)
#
# The replay reconstructs the same inputs the model saw during rollout,
# then computes log_probs of the actions that were actually taken.

model_rf = GAT(gat_config)
optimizer_rf = torch.optim.AdamW(model_rf.parameters(), lr=1e-4, weight_decay=0.1)

batch_size = 4
n_rollouts = 4
temperature = 1.0  # scalar float — safest for @torch.compile kernel
num_steps = 1000
entropy_coef = 0.01  # encourage exploration

for step in range(num_steps):
    print(":: step ", step)
    optimizer_rf.zero_grad()
    tokens, _ = train_loader.get_batch(batch_size)

    # ── 1. Sample rollouts (no grad) ──
    idx_all, rewards, advantages, history = rollout_with_advantage(
        tokens, model_rf, n=n_rollouts, K_ratio=K,
        max_iterations=MAX_ITER, memory_span=MEMORY_SPAN,
        attn_blocksize=ATTN_BS, temperature=temperature)

    # ── 2. Replay with grad to get log_probs ──
    nl_vocab = model_rf.vocab_sizes[0].item()
    data_len = tokens.shape[1]
    insert_mask = infer_rythmic_insert_mask(tokens, K, model_rf.vocab_sizes[0])
    expanded = insert_tokens(tokens, insert_mask, nl_vocab)[:, :data_len]
    repeat_data = expanded.repeat_interleave(n_rollouts, dim=0)

    recursion_mask = (repeat_data >= nl_vocab)
    recursion_mask[:, 0] = False
    predict_mask = torch.roll(recursion_mask, -1, dims=1)
    predict_mask[:, -1] = False

    n_abs = recursion_mask[0].sum().item()
    temp = max(temperature, 0.01)

    total_log_probs = torch.zeros(n_rollouts, device=device)
    total_entropy = torch.zeros(n_rollouts, device=device)

    for it in range(MAX_ITER):
        # Input: initial expanded (it=0) or previous iteration output (it>0)
        if it == 0:
            inp = repeat_data.clone()
        else:
            inp = history[it - 1].detach().clone()

        # Forward WITH grad
        _, logits = model_rf.forward(inp, MEMORY_SPAN, ATTN_BS)

        # Extract logits at positions predicting abstract tokens
        abs_logits = logits[predict_mask]  # (n_rollouts * n_abs, V)

        # Slice to ONLY abstract token logits — avoids -inf masking
        # which causes 0 * (-inf) = NaN in entropy computation
        abs_only = abs_logits[:, nl_vocab + 1:]  # (n_rollouts * n_abs, ABS_VOCAB)

        log_probs = F.log_softmax(abs_only / temp, dim=-1)
        probs = log_probs.exp()

        # Tokens that were actually sampled in this iteration
        sampled_tokens = history[it].detach()[recursion_mask]  # (n_rollouts * n_abs,)
        # Remap to abstract-only index space
        sampled_idx = sampled_tokens - (nl_vocab + 1)

        # Log-prob of sampled actions
        token_log_probs = log_probs.gather(1, sampled_idx.unsqueeze(1)).squeeze(1)
        total_log_probs += token_log_probs.view(n_rollouts, n_abs).sum(dim=1)

        # Entropy for exploration bonus (no NaN — all values are finite)
        ent = -(probs * log_probs).sum(dim=-1)
        total_entropy += ent.view(n_rollouts, n_abs).sum(dim=1)

    # ── 3. Policy gradient loss ──
    pg_loss = -(advantages.detach() * total_log_probs).mean()
    ent_loss = -entropy_coef * total_entropy.mean() # -> entropy regularizaiotn, I removed this from the main loss
    loss = pg_loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_rf.parameters(), 1.0)
    optimizer_rf.step()

    if step % 10 == 0:
        with torch.no_grad():
            vt, _ = val_loader.get_batch(batch_size)
            vs, vr, _, _ = rollout_and_select(
                vt, model_rf, n=1, K_ratio=K, max_iterations=MAX_ITER,
                memory_span=MEMORY_SPAN, attn_blocksize=ATTN_BS,
                temperature=0.01)
            ap, ti = compute_external_targets(vs[0], NL_VOCAB, ABS_VOCAB, K)
            match = (vs[0, ap] == ti).float().mean().item() if len(ap) > 0 else 0.0
        print(f"step {step:4d} | pg={pg_loss.item():.3f} ent={ent_loss.item():.3f} "
              f"loss={loss.item():.3f} | mean_r={rewards.mean().item():.2f} | "
              f"val_match={match*100:.1f}%")

:: step  0
step    0 | pg=0.000 ent=-0.650 loss=0.000 | mean_r=-4.35 | val_match=0.0%
:: step  1
:: step  2
:: step  3
:: step  4
:: step  5
:: step  6
:: step  7
:: step  8
:: step  9
:: step  10
step   10 | pg=-0.001 ent=-0.650 loss=-0.001 | mean_r=-4.42 | val_match=16.7%
:: step  11
:: step  12
:: step  13
:: step  14
:: step  15
:: step  16
:: step  17
:: step  18
:: step  19
:: step  20
step   20 | pg=-0.013 ent=-0.650 loss=-0.013 | mean_r=-4.15 | val_match=0.0%
:: step  21
:: step  22
:: step  23
:: step  24
:: step  25
:: step  26
:: step  27
:: step  28
:: step  29
:: step  30
step   30 | pg=0.003 ent=-0.650 loss=0.003 | mean_r=-4.48 | val_match=8.3%
:: step  31
:: step  32
:: step  33
:: step  34
:: step  35
:: step  36
:: step  37
:: step  38
:: step  39
:: step  40
step   40 | pg=-0.003 ent=-0.650 loss=-0.003 | mean_r=-4.00 | val_match=0.0%
:: step  41
:: step  42
:: step  43
:: step  44
:: step  45
:: step  46
:: step  47
:: step  48
:: step  49
:: step  50
step   50 | pg=-

In [6]:
evaluate_model(model, val_loader, n_batches=100, batch_size=4) # reinforce algorithm result is even worse than the best of N mechanism
# match rate 6.5%
# - much worse (despite having diverse vocabulary)

=== Evaluation (100 batches, 1200 abstract tokens) ===
  match_rate = 7.2%
  mean_reward = -4.19
  --- predicted distribution ---
    token 50258 ( 1):   73 (  6.1%)
    token 50259 ( 2):   86 (  7.2%)
    token 50260 ( 3):   80 (  6.7%)
    token 50261 ( 4):   80 (  6.7%)
    token 50262 ( 5):   83 (  6.9%)
    token 50263 ( 6):   82 (  6.8%)
    token 50264 ( 7):   88 (  7.3%)
    token 50265 ( 8):   92 (  7.7%)
    token 50266 ( 9):   78 (  6.5%)
    token 50267 (10):   69 (  5.8%)
    token 50268 (11):   70 (  5.8%)
    token 50269 (12):   78 (  6.5%)
    token 50270 (13):   89 (  7.4%)
    token 50271 (14):   78 (  6.5%)
    token 50272 (15):   74 (  6.2%)
  --- target distribution ---
    token 50257 ( 0):    2 (  0.2%)
    token 50258 ( 1):    7 (  0.6%)
    token 50259 ( 2):   21 (  1.8%)
    token 50260 ( 3):   44 (  3.7%)
    token 50261 ( 4):   84 (  7.0%)
    token 50262 ( 5):  121 ( 10.1%)
    token 50263 ( 6):  153 ( 12.8%)
    token 50264 ( 7):  152 ( 12.7%)
    token 50

(0.0725,
 -4.194166648387909,
 Counter({50265: 92,
          50270: 89,
          50264: 88,
          50259: 86,
          50262: 83,
          50263: 82,
          50261: 80,
          50260: 80,
          50266: 78,
          50271: 78,
          50269: 78,
          50272: 74,
          50258: 73,
          50268: 70,
          50267: 69}),
 Counter({50265: 179,
          50266: 162,
          50263: 153,
          50264: 152,
          50262: 121,
          50267: 116,
          50261: 84,
          50268: 76,
          50269: 51,
          50260: 44,
          50270: 26,
          50259: 21,
          50258: 7,
          50271: 6,
          50257: 2}))